# 🧪 HyDE — Hypothetical Document Embeddings
## RAG with LangChain + Ollama + ChromaDB

---

## 🧠 What is HyDE?

**HyDE = Hypothetical Document Embeddings**

A retrieval technique where instead of embedding the *user's question*, you ask the LLM to first generate a *hypothetical answer document*, and then embed **that** for retrieval.

**The Core Insight:**  
Questions and answers live in *different vector spaces*.  
A question like *"What is the leave policy?"* looks nothing like a policy document paragraph.  
But a *hypothetical answer* — even if factually incorrect — looks a lot like a real answer paragraph.

```
Without HyDE:
  "What is the leave policy?"  →  embed  →  search  →  chunks
   (question-shaped vector)                  (answer-shaped chunks)
   ↑ semantic mismatch ↑

With HyDE:
  "What is the leave policy?"  →  LLM generates hypothetical answer
  "Employees are entitled to 18 days of annual leave per year..."  →  embed  →  search  →  chunks
   (answer-shaped vector)                                                        (answer-shaped chunks)
   ↑ much better alignment ↑
```

**Analogy:**  
Imagine searching for a song but you don't know the title. You *hum the tune* instead of describing it in words — you give the search system something that *sounds like* what you're looking for, not just a description of it. HyDE does the same: it gives the vector store something that *looks like* the answer it's searching for.

---

## 🗺️ Full Pipeline Overview

```
┌─────────────────────────────────────────────────────────────┐
│                   INDEXING PHASE (run once)                 │
│                                                             │
│  Raw Text → TextLoader → Chunks → Embeddings → ChromaDB    │
└─────────────────────────────────────────────────────────────┘
              ↓  (query time — the HyDE flow)
┌─────────────────────────────────────────────────────────────┐
│                   QUERYING PHASE (per question)             │
│                                                             │
│  User Question                                              │
│         │                                                   │
│  [Step 7] HyDE Chain (LLM)                                  │
│         │  → Generates a hypothetical answer paragraph      │
│         │                                                   │
│  [Step 8a] Retrieve using ORIGINAL question   (compare)    │
│  [Step 8b] Retrieve using HYPOTHETICAL DOC    (for answer) │
│         │                                                   │
│  [Step 8c] Build prompt with HyDE context                  │
│         │                                                   │
│  [Step 8d] LLM generates final grounded answer             │
└─────────────────────────────────────────────────────────────┘
```

> 💡 **Key Insight:** The hypothetical document is *thrown away* after retrieval — it's only a search probe. The final answer is always grounded in real retrieved chunks, never in the hypothetical text.

## 🔄 Progression: How HyDE Fits in the RAG Evolution

| Approach | What goes into the vector search? | Strength |
|---|---|---|
| **Basic RAG** | Raw user question | Simple, fast |
| **Query Rewriting** | LLM-rephrased search query | Better keyword alignment |
| **HyDE** ⭐ | LLM-generated hypothetical answer paragraph | Best semantic alignment |

Each step moves the search vector *closer to what the answer looks like* — which is what the document chunks look like.

## 📦 Prerequisites & Installation

Same stack as the previous notebooks:

1. **Ollama running** → https://ollama.com
2. **Models pulled:**
   ```bash
   ollama pull nomic-embed-text
   ollama pull gpt-oss:120b-cloud
   ```
3. **Packages installed:**
   ```bash
   pip install langchain langchain-community langchain-ollama langchain-chroma langchain-core chromadb
   ```
4. **Sample file at** `docs/company_policy.txt`

---
## 📚 Step 0 — Imports

Identical imports to the Query Rewriting notebook — HyDE reuses the same building blocks, just changes *what the LLM is asked to generate*.

| Import | Role |
|---|---|
| `TextLoader` | Reads `.txt` file from disk |
| `RecursiveCharacterTextSplitter` | Splits text into overlapping chunks |
| `OllamaEmbeddings` | Converts text → dense vectors |
| `Chroma` | Local vector database |
| `ChatOllama` | Local LLM |
| `PromptTemplate` | Parameterised prompt builder for the HyDE generator |

In [1]:
from langchain_community.document_loaders.text import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate

print("✅ All imports successful")

C:\Users\sr43993\AppData\Local\Temp\ipykernel_9948\1327660044.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


✅ All imports successful


---
## 📄 Steps 1–6 — Indexing Pipeline

Identical to the previous two notebooks. Kept concise — focus is on Step 7 (HyDE).

> 📖 For deep-dive explanations of each step, refer to the **Basic RAG notebook**.

In [2]:
# ── Step 1: Load Document ────────────────────────────────────────────────────
print("Loading document...\n")

loader = TextLoader("docs/company_policy.txt", encoding="utf-8")
documents = loader.load()

print(f"✅ Loaded {len(documents)} document(s)")
print(f"\n📄 Preview: {documents[0].page_content[:200]}")

Loading document...

✅ Loaded 1 document(s)

📄 Preview: Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.


In [3]:
# ── Step 2: Split into Chunks ────────────────────────────────────────────────
print("Splitting document into chunks...\n")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)
chunks = text_splitter.split_documents(documents)

print(f"✅ Total Chunks: {len(chunks)}")
for i, chunk in enumerate(chunks[:3]):
    print(f"\n🔹 Chunk {i+1}: {chunk.page_content}")
    print("-" * 50)

Splitting document into chunks...

✅ Total Chunks: 1

🔹 Chunk 1: Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period before resignation.

Annual leave balance is 24 days.

Work from home is allowed twice a week.
--------------------------------------------------


In [4]:
# ── Step 3: Embedding Model ──────────────────────────────────────────────────
print("Loading Embedding Model...\n")

embeddings = OllamaEmbeddings(model="nomic-embed-text")

test_vec = embeddings.embed_query("test")
print(f"✅ Embedding Model Ready — vector dim: {len(test_vec)}")

Loading Embedding Model...

✅ Embedding Model Ready — vector dim: 768


In [5]:
# ── Step 4: Create & Persist Vector Store ────────────────────────────────────
print("Creating Chroma Vector Store...\n")

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print(f"✅ Vector Store Ready — {vectorstore._collection.count()} vectors stored")

Creating Chroma Vector Store...

✅ Vector Store Ready — 5 vectors stored


In [6]:
# ── Step 5: Retriever ────────────────────────────────────────────────────────
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
print("✅ Retriever Ready")

✅ Retriever Ready


In [7]:
# ── Step 6: Load LLM ─────────────────────────────────────────────────────────
print("Loading LLM...\n")

llm = ChatOllama(
    model="gpt-oss:120b-cloud",
    temperature=0   # Deterministic — required for grounded RAG answers
)

print("✅ LLM Ready")

Loading LLM...

✅ LLM Ready


---
## 🧪 Step 7 — Build the HyDE Chain

This is the heart of the notebook — and conceptually the simplest change with the biggest retrieval impact.

### What the HyDE prompt asks the LLM to do:

```
User asks: "What is the leave policy?"

HyDE prompt instructs LLM:
  "Generate a paragraph that would likely ANSWER this question"

LLM generates (hypothetical — may be factually wrong):
  "Employees are entitled to 18 days of paid annual leave per year.
   Leave must be applied for at least 2 weeks in advance through
   the HR portal..."

This paragraph is now embedded and used for search.
```

### Why does a *wrong* hypothetical document still help?

Because embedding similarity is about **style and vocabulary**, not just facts.  
A hypothetical paragraph about leave policy will use the same domain terms (*"annual leave"*, *"HR portal"*, *"entitlement"*) as the real policy document — even if the numbers are wrong. The vector will land in the right neighborhood of the embedding space.

```
Vector Space (simplified 2D view):

         "What is leave policy?"  ← question vector
                    ↑
                (gap)
                    ↓
  "Employees get 18 days leave..." ← hypothetical vector  ←→  "Annual leave: 20 days..." ← real chunk
  (wrong facts, right vocabulary)                               (this is what we want to find)
```

> ⚠️ **Critical Rule:** The hypothetical document is a **search probe only**. It is NEVER shown to the user and NEVER used in the final answer prompt. The final answer is always grounded in the real retrieved chunks.

> 💡 **Production Note:** Consider using a *smaller, faster model* (e.g. `llama3.2:3b`) for generating the hypothetical document, since it doesn't need to be factually accurate — just stylistically similar to the domain.

In [8]:
# ── HyDE Prompt ───────────────────────────────────────────────────────────────
# Instructs the LLM to generate a hypothetical answer paragraph
# The output will be embedded and used as the search query — NOT as the answer

hyde_prompt = PromptTemplate.from_template(
    """
You are helping a retrieval system.

Generate a short hypothetical document paragraph that would
likely answer the user's question.

Write in the style of a formal policy or reference document.
It does NOT need to be factually correct — focus on style and vocabulary.

Question:
{question}

Hypothetical Document:
"""
)

# ── HyDE Chain (LCEL pipe) ───────────────────────────────────────────────────
# Fill {question} → send to LLM → return hypothetical paragraph
hyde_chain = hyde_prompt | llm

print("✅ HyDE Chain Ready")

# ── Sanity Check: See what a hypothetical document looks like ─────────────────
sample_q = "What is the leave policy?"
sample_hyde = hyde_chain.invoke({"question": sample_q})

print(f"\n🧪 Sample HyDE Test:")
print(f"\n❓ Question:")
print(f"   {sample_q}")
print(f"\n📝 Generated Hypothetical Document:")
print(sample_hyde.content.strip())
print("\n(This paragraph will be embedded and used as the search vector — not shown to the user)")

✅ HyDE Chain Ready

🧪 Sample HyDE Test:

❓ Question:
   What is the leave policy?

📝 Generated Hypothetical Document:
**Leave Policy** – The organization hereby establishes a comprehensive leave framework applicable to all full‑time, part‑time, and temporary personnel. Eligible employees accrue a minimum of fifteen (15) calendar days of paid annual leave per fiscal year, prorated on a monthly basis for periods of service less than twelve months. In addition to annual leave, the policy provisions up to ten (10) days of designated sick leave, five (5) days of bereavement leave, and twelve (12) days of parental leave per qualifying event, each subject to documentation and managerial approval. Leave requests must be submitted through the centralized Human‑Resources portal at least five (5) business days prior to the intended start date, except in cases of emergency or unforeseen illness, wherein notification shall be made at the earliest possible opportunity. Unused accrued annual leave ma

---
## 💬 Step 8 — Interactive Q&A Loop with HyDE

### The Full Flow for Every Question:

```
User's raw question
        │
        ├────────────────────────────────────────────────┐
        │                                                │
   [HyDE Chain]                                   [kept for display]
   LLM writes a hypothetical                      shown in final
   answer paragraph                               answer prompt
        │
        ├── [Retrieval A] Original question  → ChromaDB  (comparison)
        ├── [Retrieval B] Hypothetical doc   → ChromaDB  (used for answer)
        │
   Build final prompt:
   real chunks from Retrieval B + original question
        │
   LLM generates grounded answer
```

### What to observe while running:
- Compare the **ORIGINAL RETRIEVAL** vs **HyDE RETRIEVAL** chunks — HyDE should pull more on-topic chunks
- Notice that the **hypothetical document** often contains wrong facts, but still retrieves the right chunks
- The **final answer** is always based on real retrieved content, never the hypothetical text

In [9]:
print("HyDE RAG System Ready. Type 'exit' to stop.\n")
print("=" * 60)

while True:

    question = input("\n❓ Ask a question (or type 'exit'): ")

    if question.lower() == "exit":
        print("\n👋 Goodbye!")
        break

    # ── STEP 8a: GENERATE HYPOTHETICAL DOCUMENT ───────────────────────────────
    # LLM writes a fake-but-stylistically-accurate answer paragraph
    # This becomes the search vector — it is NOT the final answer
    hypothetical_response = hyde_chain.invoke({"question": question})
    hypothetical_text = hypothetical_response.content.strip()

    print("\n===== ❓ USER QUESTION =====")
    print(question)

    print("\n===== 📝 HYPOTHETICAL DOCUMENT (search probe only) =====")
    print(hypothetical_text)
    print("(⚠️ This is NOT the answer — it's only used to find the right chunks)")

    # ── STEP 8b: RETRIEVE USING ORIGINAL QUESTION ────────────────────────────
    # For side-by-side comparison — shows what vanilla RAG would have retrieved
    original_docs = retriever.invoke(question)

    print("\n===== 📄 ORIGINAL RETRIEVAL (vanilla RAG — for comparison) =====")
    for index, doc in enumerate(original_docs):
        print(f"\n  Chunk {index + 1}: {doc.page_content}")

    # ── STEP 8c: RETRIEVE USING HYPOTHETICAL DOCUMENT ────────────────────────
    # The hypothetical paragraph is embedded and matched against real chunks
    # Because it's answer-shaped, it finds answer-shaped chunks more reliably
    retrieved_docs = retriever.invoke(hypothetical_text)

    print("\n===== 📚 HyDE RETRIEVAL (used for final answer) =====")
    for index, doc in enumerate(retrieved_docs):
        print(f"\n  Chunk {index + 1}: {doc.page_content}")

    # ── STEP 8d: BUILD CONTEXT FROM REAL RETRIEVED CHUNKS ────────────────────
    # The hypothetical doc is discarded here — only real chunks go into the answer
    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    # ── STEP 8e: CRAFT FINAL ANSWER PROMPT ───────────────────────────────────
    # Original question + real context = grounded, trustworthy answer
    # The hypothetical document does NOT appear here
    prompt = f"""You are a helpful assistant.

Answer ONLY using the provided context.
If the answer is not present in the context, reply exactly with:
"I could not find that information in the document."

Context:
{context}

Question:
{question}
"""

    # ── STEP 8f: GENERATE FINAL ANSWER ───────────────────────────────────────
    response = llm.invoke(prompt)

    print("\n===== 🤖 FINAL ANSWER =====")
    print(response.content)

HyDE RAG System Ready. Type 'exit' to stop.


===== ❓ USER QUESTION =====
maternity leave

===== 📝 HYPOTHETICAL DOCUMENT (search probe only) =====
**Maternity Leave Policy – Section 4.2 (Employee Benefits Manual)**  

All full‑time employees who have completed a minimum of twelve (12) months of continuous service are entitled to paid maternity leave in accordance with the provisions set forth herein. Eligible employees may elect to commence leave up to four (4) weeks prior to the expected date of confinement and may extend the absence for a total duration not to exceed twelve (12) consecutive weeks, provided that a certified medical statement confirming the anticipated delivery date is submitted to Human Resources no later than thirty (30) calendar days before the intended start of leave. Compensation during the leave period shall be calculated at the employee’s regular base salary, subject to applicable statutory deductions and withholdings. Upon successful completion of the leave, th

---
## 🧪 Bonus — Single-Shot HyDE Query

Test any question without entering the interactive loop.

In [10]:
# ── Change this question and re-run ──────────────────────────────────────────
question = "How many sick days am I allowed per year?"

# Generate hypothetical document (search probe)
hypothetical_text = hyde_chain.invoke({"question": question}).content.strip()

# Retrieve using hypothetical document
docs = retriever.invoke(hypothetical_text)
context = "\n\n".join(doc.page_content for doc in docs)

# Generate grounded answer using real chunks
prompt = f"""Answer ONLY using the context below.
If not found, say: "I could not find that information in the document."

Context:
{context}

Question: {question}
"""
answer = llm.invoke(prompt).content

print(f"❓ Question          : {question}")
print(f"\n📝 Hypothetical Doc  : {hypothetical_text[:200]}...")
print(f"\n🤖 Answer            : {answer}")

❓ Question          : How many sick days am I allowed per year?

📝 Hypothetical Doc  : **Employee Sick‑Leave Policy – Annual Entitlement**

In accordance with the Company Employee Handbook (Section 4.2, “Paid Sick Leave”), each regular full‑time employee shall be entitled to **twelve (1...

🤖 Answer            : I could not find that information in the document.


---
## 🔬 Bonus — HyDE vs Vanilla RAG: Side-by-Side Retrieval Comparison

Run multiple questions and compare which chunks each approach retrieves.  
This is the **best demo cell** for training audiences — it makes the difference visible.

In [11]:
test_questions = [
    "Can I take time off?",
    "What happens if I arrive late?",
    "Do I get paid if I'm sick?",
    "What is the notice period if I resign?",
]

for q in test_questions:
    print(f"\n{'='*65}")
    print(f"❓ Question: {q}")

    # Vanilla RAG retrieval
    vanilla_chunks = retriever.invoke(q)
    print(f"\n📄 Vanilla RAG retrieved:")
    for doc in vanilla_chunks:
        print(f"   → {doc.page_content[:100]}...")

    # HyDE retrieval
    hyp_text = hyde_chain.invoke({"question": q}).content.strip()
    hyde_chunks = retriever.invoke(hyp_text)
    print(f"\n🧪 HyDE retrieved:")
    for doc in hyde_chunks:
        print(f"   → {doc.page_content[:100]}...")

    # Are they the same?
    vanilla_set = {doc.page_content for doc in vanilla_chunks}
    hyde_set = {doc.page_content for doc in hyde_chunks}
    different = vanilla_set != hyde_set
    print(f"\n  {'⭐ HyDE retrieved DIFFERENT chunks' if different else '= Same chunks retrieved'}")


❓ Question: Can I take time off?

📄 Vanilla RAG retrieved:
   → Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period bef...
   → Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period bef...

🧪 HyDE retrieved:
   → Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period bef...
   → Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period bef...

  = Same chunks retrieved

❓ Question: What happens if I arrive late?

📄 Vanilla RAG retrieved:
   → Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period bef...
   → Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period bef...

🧪 HyDE retrieved:
   → Employees are eligible for 6 months maternity leave.

Employees must serve 60 days notice period bef...
   → Employees are eligible for 6 months m

---
## 🎯 Key Takeaways

| Concept | What to Remember |
|---|---|
| **HyDE core idea** | Embed a hypothetical *answer*, not the *question* |
| **Why wrong facts are OK** | Vector similarity is about vocabulary & style, not factual accuracy |
| **Hypothetical doc is discarded** | It's a search probe only — never used in the final answer |
| **LLM called twice** | Once to generate the hypothetical doc, once to generate the final answer |
| **Original question preserved** | Always used in the final answer prompt — user intent stays intact |
| **Best use case** | Complex or abstractly worded questions where direct embedding underperforms |

---

## 🔄 The Three RAG Approaches — Side by Side

```
Basic RAG:
  User question ──────────────────────────────→ Retriever → LLM → Answer
  (question vector — semantic mismatch risk)

Query Rewriting:
  User question → LLM rewrites to search terms → Retriever → LLM → Answer
  (keyword-optimized — better term matching)

HyDE:
  User question → LLM writes hypothetical answer → Retriever → LLM → Answer
  (answer-shaped vector — best semantic alignment with document chunks)
```

---

## ⚖️ HyDE Trade-offs

| ✅ Strengths | ⚠️ Weaknesses |
|---|---|
| Best retrieval for vague/abstract questions | 2× LLM calls = higher latency |
| Bridges question-answer vocabulary gap | Hypothetical doc may mislead if domain is unfamiliar to LLM |
| No need to engineer search queries | Harder to debug than query rewriting |
| Works well for long-form policy/technical docs | Overkill for simple keyword questions |

---

## 🚀 Next Steps to Explore

1. **Multi-query HyDE** — Generate 3 different hypothetical documents per question and merge retrieved results
2. **Combine HyDE + Reranking** — Use a cross-encoder to rerank HyDE-retrieved chunks by actual relevance
3. **HyDE with smaller model** — Use `llama3.2:3b` for hypothetical generation, large model for answering
4. **RAGAS evaluation** — Measure whether HyDE actually improves faithfulness and context recall vs Vanilla RAG
5. **Step-back prompting** — Abstract the question further before HyDE for even broader retrieval coverage